In [20]:
import librosa
import numpy as np
import statistics

In [33]:
def compute_swing(beats, onsets, min_swing: float = 0.2, max_swing: float = 0.9):
    """
    Estimate swing ratio from beat intervals.

    Returns:
        swing_score
        stability
        interval_count
    """

    swing_values = []

    for i in range(len(beats) - 1):
        b0 = beats[i]
        b1 = beats[i + 1]
        duration = b1 - b0

        if duration <= 0:
            continue

        midpoint = b0 + duration * 0.5

        candidates = onsets[
            (onsets > b0 + duration * 0.15)
            & (onsets < b1 - duration * 0.15)
        ]

        if len(candidates) == 0:
            continue

        distances = np.abs(candidates - midpoint)
        idx = np.argmin(distances)
        onset = candidates[idx]
        normalized_position = (onset - b0) / duration

        # Reject unlikely values.
        if normalized_position < min_swing:
            continue

        if normalized_position > max_swing:
            continue

        # Reject intervals where several onsets compete.
        competing = np.sum(
            np.abs(candidates - midpoint)
            < duration * 0.10
        )

        if competing > 3:
            continue

        swing_values.append(float(normalized_position))

    if len(swing_values) < 10:
        raise RuntimeError(
            "Too few valid intervals found."
        )

    swing = statistics.median(swing_values)
    stability = statistics.median(
        [abs(x - swing) for x in swing_values]
    )

    return swing, stability, len(swing_values), swing_values

In [29]:
def analyze_file(audio_file, offset: float | None = None, duration: float | None = None):
    data, rate = librosa.load(audio_file, offset=offset, duration=duration)
    tempo, beats = librosa.beat.beat_track(y=data, sr=rate, units='time')
    onsets = librosa.onset.onset_detect(y=data, sr=rate, units='time')

    return compute_swing(beats, onsets)

In [34]:
analyze_file("The James Hunter Six - Light of My Life - SDGT-2024-07.mp3")

(0.6521739130434779,
 0.029644268774693727,
 270,
 [0.375,
  0.29032258064516125,
  0.625,
  0.6818181818181821,
  0.6363636363636366,
  0.6521739130434779,
  0.652173913043479,
  0.6521739130434779,
  0.6521739130434779,
  0.652173913043479,
  0.681818181818181,
  0.6666666666666662,
  0.6363636363636357,
  0.6363636363636364,
  0.6086956521739139,
  0.6086956521739152,
  0.6818181818181821,
  0.6666666666666666,
  0.695652173913044,
  0.6521739130434779,
  0.6521739130434779,
  0.695652173913043,
  0.5909090909090893,
  0.4347826086956509,
  0.36363636363636426,
  0.782608695652173,
  0.6956521739130463,
  0.565217391304351,
  0.8095238095238133,
  0.31818181818182134,
  0.30434782608695826,
  0.36363636363636426,
  0.3333333333333333,
  0.29166666666666907,
  0.6521739130434792,
  0.6363636363636357,
  0.624999999999996,
  0.6363636363636357,
  0.666666666666671,
  0.6521739130434792,
  0.6666666666666666,
  0.6521739130434792,
  0.6666666666666645,
  0.6086956521739099,
  0.6521739

In [35]:
analyze_file("JJ Grey & Mofro feat. Mofro - On Fire - AMZN-2024-05.mp3")

(0.4999999999999941,
 0.018518518518512078,
 394,
 [0.4615384615384614,
  0.5,
  0.5000000000000002,
  0.5,
  0.48148148148148145,
  0.5,
  0.5185185185185175,
  0.4814814814814818,
  0.46153846153846223,
  0.48000000000000037,
  0.46153846153846073,
  0.4814814814814818,
  0.46153846153846007,
  0.4999999999999985,
  0.4999999999999985,
  0.4814814814814797,
  0.4615384615384593,
  0.4999999999999985,
  0.4999999999999985,
  0.4999999999999985,
  0.48148148148148107,
  0.5000000000000014,
  0.5,
  0.48148148148148107,
  0.46153846153846223,
  0.4999999999999985,
  0.5,
  0.49999999999999706,
  0.4814814814814825,
  0.5,
  0.49999999999999706,
  0.5185185185185175,
  0.4615384615384579,
  0.49999999999999706,
  0.42307692307692174,
  0.4615384615384606,
  0.4615384615384579,
  0.5384615384615393,
  0.5185185185185175,
  0.5,
  0.49999999999999706,
  0.5384615384615362,
  0.5185185185185204,
  0.4800000000000012,
  0.5,
  0.4400000000000002,
  0.4814814814814825,
  0.4814814814814825,
 